In [0]:
import pandas as pd

from pyspark.sql import functions as F

In [0]:
# read current service group mappings
current = spark.table("opsanalytics_adb_workspace01.capacity_modeling.service_groups").toPandas()

display(current)

In [0]:
oracle_jdbc_url = dbutils.secrets.get(
    scope="oao_secrets",
    key="ORACLE_JDBC_URL",
)

oracle_password = dbutils.secrets.get(
    scope="oao_secrets",
    key="OAO_PRODUCTION",
)

clarity_dep_ref = (
    spark.read
    .format("jdbc")
    .option("url", oracle_jdbc_url)
    .option(
        "dbtable",
        "DASHBD_USER.CLARITY_DEP_REF",
    )
    .option("user", "OAO_PRODUCTION")
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("oracle.net.ssl_server_dn_match", "true")
    .option("fetchsize", "10000")
    .load()
    .toPandas()
)

display(clarity_dep_ref)

In [0]:
# filter current df 
current = current.loc[
    current["VALID_TO"].isna() &
    ~current["EXTERNAL_NAME"].str.contains("EMERGENCY", na=False)
]

#merge current and new tables
compare = current.merge(
    clarity_dep_ref,
    left_on="EPIC_DEPT_ID",
    right_on="DEPARTMENT_ID",
    how="outer",
    suffixes=("", "_clarity"),
)

# Trim whitespace before comparing
cols_to_trim = [
    "LOC_NAME", "LOC_NAME_clarity",
    "EXTERNAL_NAME", "EXTERNAL_NAME_clarity",
    "SERVICE_GROUP", "SERVICE_GROUP_clarity",
]
compare[cols_to_trim] = compare[cols_to_trim].apply(lambda col: col.str.strip())

# Each column will contain True or False for every row
compare["LOCATION_CHANGE"] = compare["LOC_NAME"] != compare["LOC_NAME_clarity"]
compare["NAME_CHANGE"] = compare["EXTERNAL_NAME"] != compare["EXTERNAL_NAME_clarity"]
compare["GROUP_CHANGE"] = compare["SERVICE_GROUP"] != compare["SERVICE_GROUP_clarity"]

display(compare)

In [0]:
change = compare.loc[
    compare["LOC_NAME"].notnull() 
].loc[
    compare["LOCATION_CHANGE"]
    | compare["NAME_CHANGE"]
    | compare["GROUP_CHANGE"]
]

change = change[
    ["EPIC_DEPT_ID"] + [
        col for col in change.columns
        if any(name in col for name in ("LOC_NAME", "EXTERNAL_NAME", "SERVICE_GROUP", "CHANGE"))
    ]
]

if change.empty :
    print("There were no changes to Epic Mappings")
else :
    change = change.assign(
    LOC_NAME=change["LOC_NAME_new"],
    EXTERNAL_NAME=change["EXTERNAL_NAME_new"],
    SERVICE_GROUP=change["SERVICE_GROUP_new"],
    VALID_FROM=pd.Timestamp.today().normalize(),
    VALID_TO=pd.NaT,
    )[
    ["LOC_NAME", "EPIC_DEPT_ID", "EXTERNAL_NAME", "SERVICE_GROUP", "VALID_FROM", "VALID_TO"]
    ]

    display(change)


In [0]:
# get epic deps that are not in current mappings
new_deps = (
    compare.loc[
    compare["LOC_NAME"].isnull(), "EPIC_DEPT_ID"]
    .dropna()
    .unique()
    .tolist()
)

# check in new deps have bed charges
check = (
    spark.table("opsanalytics_adb_workspace01.capacity_modeling.oe_charge_detail")
    .filter(F.col("BILLING_CAT_DESC") == "BED CHARGES")
    .filter(F.col("SERVICE_DATE") >= 20240601)
    .filter(~F.col("EPIC_DEPT_ID").isin("8503016", "8503017"))
    .filter(F.col("EPIC_DEPT_ID").cast("string").isin(new_deps))
    .select("EPIC_DEPT_ID")
    .distinct()
    .toPandas()
)

if check.empty :
    print("There were no new departments with bed charges")
else :
    display(check)